# 02 — Funding Rate Term Structure

Analyse the term structure of funding rates across different time horizons.
Identify regime changes and cross-symbol opportunities.

In [ ]:
import sys

sys.path.insert(0, '..')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.downloader import FundingRateDownloader
from research.term_structure import FundingTermStructure

In [ ]:
funding = FundingRateDownloader.load_cached_data()
print(f"Loaded {len(funding)} records")

In [ ]:
# Compute term structure
ts = FundingTermStructure()
ts_df = ts.compute(funding)
print(f"Term structure records: {len(ts_df)}")
cols = ['timestamp', 'symbol', 'exchange', 'rate_short_ann',
          'rate_medium_ann', 'rate_long_ann', 'slope', 'regime']
ts_df[cols].tail(10)


In [ ]:
# BTC term structure over time
btc_ts = ts_df[ts_df['symbol'] == 'BTC/USDT:USDT'].copy()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Funding Rates by Horizon', 'Term Structure Slope'])

pairs = [
    ('rate_short_ann', '1-day'),
    ('rate_medium_ann', '7-day'),
    ('rate_long_ann', '30-day'),
]
for col, name in pairs:
    fig.add_trace(go.Scatter(x=btc_ts['timestamp'], y=btc_ts[col], name=name), row=1, col=1)

fig.add_trace(go.Scatter(x=btc_ts['timestamp'], y=btc_ts['slope'] * 3 * 365,
                         name='Slope (ann.)', fill='tozeroy'), row=2, col=1)
fig.update_layout(title='BTC/USDT Funding Rate Term Structure', height=600)
fig.show()

In [ ]:
# Regime distribution
regime_counts = ts_df.groupby(['symbol', 'regime']).size().reset_index(name='count')
fig = px.bar(regime_counts, x='symbol', y='count', color='regime',
             title='Term Structure Regime Distribution by Symbol')
fig.show()

In [ ]:
# Cross-symbol term structure snapshot (latest)
snapshot = ts.cross_symbol_term_structure(funding)
if not snapshot.empty:
    fig = px.bar(snapshot.head(20), x='symbol', y='slope',
                 color='rate_short_ann',
                 title='Current Term Structure Slope by Symbol (Top 20)')
    fig.show()
    snapshot[['symbol', 'exchange', 'rate_short_ann', 'rate_long_ann', 'slope', 'regime']].head(20)